# Hospital Readmission Risk Prediction
## 03. Experiment Tracking — Scenario 2: XGBoost/LightGBM + Optuna

Per proposal 4.3: **XGBoost** and **LightGBM** with **Optuna** hyperparameter search. A model is promoted only if it exceeds the baseline on validation **PR-AUC**.

### MLflow setup (centralized tracking server)
- **Tracking**: Connect to scenario-1's UI server (port 5001); run scenario-1 first to launch it
- **UI**: http://localhost:5001 (same as scenario-1; run scenario-1 first to launch UI)
- **Metrics**: PR-AUC (primary), ROC-AUC, Recall at K

In [1]:
%pip install mlflow pandas scikit-learn xgboost lightgbm optuna "optuna-integration[mlflow]" seaborn matplotlib

Note: you may need to restart the kernel to use updated packages.


In [2]:
import mlflow
mlflow.set_tracking_uri("http://127.0.0.1:5001")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

Tracking URI: http://127.0.0.1:5001


## Load Data & Prepare Features
Same logic as scenario-1 and 02: patient-level split, feature engineering.

In [3]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import average_precision_score, roc_auc_score, confusion_matrix
from sklearn.model_selection import train_test_split
import xgboost as xgb
import lightgbm as lgb
import optuna
from optuna.integration import MLflowCallback

DATA_PATH = '../data/diabetic_data.csv'
TARGET = 'target'
SEED = 42

def read_data(path, limit=None):
    df = pd.read_csv(path)
    df['target'] = df['readmitted'].isin(['30', '<30']).astype(int)
    if 'weight' in df.columns:
        df = df.drop(columns=['weight'])
    for col in ['medical_specialty', 'payer_code']:
        if col in df.columns:
            df[col] = df[col].fillna('Unknown').replace('?', 'Unknown')
    df['age'] = df['age'].fillna('[50-60)')
    df['gender'] = df['gender'].fillna('Unknown')
    df['change'] = df['change'].fillna('No')
    df['diabetesMed'] = df['diabetesMed'].fillna('No')
    for col in ['A1Cresult', 'max_glu_serum']:
        if col not in df.columns:
            df[col] = 'not_tested'
        else:
            df[col] = df[col].fillna('None').replace('None', 'not_tested').astype(str)
    if 'race' in df.columns:
        df['race'] = df['race'].fillna('Unknown').replace('?', 'Unknown').astype(str)
    for col in ['number_emergency', 'number_inpatient', 'number_outpatient']:
        df[col] = pd.to_numeric(df.get(col, 0), errors='coerce').fillna(0).astype(int)
    df['care_intensity'] = df['number_emergency'] + df['number_inpatient'] + df['number_outpatient']
    df['medication_changed'] = (df['change'] == 'Ch').astype(int)
    for col in ['num_lab_procedures', 'num_procedures', 'num_medications', 'number_diagnoses']:
        df[col] = pd.to_numeric(df.get(col, 0), errors='coerce').fillna(0).astype(int)
    if limit:
        df = df.sample(n=min(limit, len(df)), random_state=SEED)
    return df

FEATURE_COLS = ['time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications',
    'number_emergency', 'number_inpatient', 'number_outpatient', 'number_diagnoses', 'care_intensity',
    'admission_type_id', 'discharge_disposition_id', 'admission_source_id',
    'age', 'gender', 'race', 'change', 'diabetesMed', 'medication_changed', 'A1Cresult', 'max_glu_serum']

df = read_data(DATA_PATH, limit=50_000)
FEATURE_COLS = [c for c in FEATURE_COLS if c in df.columns]
patient_target = df.groupby('patient_nbr')[TARGET].max()
train_patients, val_patients = train_test_split(
    patient_target.index.tolist(), test_size=0.2, random_state=SEED, stratify=patient_target.values
)
df_train = df[df['patient_nbr'].isin(train_patients)]
df_val = df[df['patient_nbr'].isin(val_patients)]

train_dicts = df_train[FEATURE_COLS].to_dict(orient='records')
val_dicts = df_val[FEATURE_COLS].to_dict(orient='records')
dv = DictVectorizer()
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)
y_train = df_train[TARGET].values
y_val = df_val[TARGET].values

scale_pos_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
print(f'Train: {X_train.shape[0]:,} rows; Val: {X_val.shape[0]:,} rows')
print(f'Positive rate (train): {y_train.mean():.2%}; scale_pos_weight: {scale_pos_weight:.2f}')

Train: 40,030 rows; Val: 9,970 rows
Positive rate (train): 11.15%; scale_pos_weight: 7.97


## Optuna + XGBoost
Hyperparameter search over learning rate, max_depth, n_estimators. Optimize **PR-AUC** (primary metric).

In [4]:
def recall_at_k(y_true, y_proba, k=0.2):
    n = int(len(y_true) * k)
    top_idx = np.argsort(y_proba)[::-1][:n]
    return y_true[top_idx].sum() / max(y_true.sum(), 1)

def objective_xgb(trial):
    params = {
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'n_estimators': trial.suggest_int('n_estimators', 50, 200),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'scale_pos_weight': scale_pos_weight,
        'random_state': SEED,
        'eval_metric': 'aucpr',
    }
    model = xgb.XGBClassifier(**params)
    model.fit(X_train, y_train, verbose=False)
    y_proba = model.predict_proba(X_val)[:, 1]
    pr_auc = average_precision_score(y_val, y_proba)
    return pr_auc

mlflow.set_experiment("hospital-readmission-xgboost")
mlflow_callback = MLflowCallback(
    tracking_uri=mlflow.get_tracking_uri(),
    metric_name='pr_auc',
)

study_xgb = optuna.create_study(direction='maximize', study_name='xgboost-optuna')
study_xgb.optimize(objective_xgb, n_trials=5, callbacks=[mlflow_callback], show_progress_bar=True)
print(f'Best XGBoost PR-AUC: {study_xgb.best_value:.4f}')

2026/03/10 21:38:16 INFO mlflow.tracking.fluent: Experiment with name 'hospital-readmission-xgboost' does not exist. Creating a new experiment.
/var/folders/_y/x_pznj0x0m16xm40mxczxrl00000gn/T/ipykernel_44369/934256255.py:24: ExperimentalWarning: MLflowCallback is experimental (supported from v1.4.0). The interface can change in the future.
  mlflow_callback = MLflowCallback(
[I 2026-03-10 21:38:16,668] A new study created in memory with name: xgboost-optuna


  0%|          | 0/5 [00:00<?, ?it/s]

2026/03/10 21:38:16 INFO mlflow.tracking.fluent: Experiment with name 'xgboost-optuna' does not exist. Creating a new experiment.


[I 2026-03-10 21:38:16,868] Trial 0 finished with value: 0.20820419017920525 and parameters: {'learning_rate': 0.05337708728258888, 'max_depth': 3, 'n_estimators': 54, 'subsample': 0.9918271220367518, 'colsample_bytree': 0.8241983062773723}. Best is trial 0 with value: 0.20820419017920525.
🏃 View run 0 at: http://127.0.0.1:5001/#/experiments/3/runs/5f3e8e21001644abaf78fa80bdcd76ea
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/3
[I 2026-03-10 21:38:17,490] Trial 1 finished with value: 0.21074282667379451 and parameters: {'learning_rate': 0.01923394468527933, 'max_depth': 6, 'n_estimators': 72, 'subsample': 0.6317447007461545, 'colsample_bytree': 0.6607419252449758}. Best is trial 1 with value: 0.21074282667379451.
🏃 View run 1 at: http://127.0.0.1:5001/#/experiments/3/runs/38e40a0aead245bab7151546d39dbaed
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/3
[I 2026-03-10 21:38:18,229] Trial 2 finished with value: 0.20715209619061786 and parameters: {'learning_rate': 0

## Optuna + LightGBM
Same setup for LightGBM.

In [5]:
def objective_lgb(trial):
    params = {
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'n_estimators': trial.suggest_int('n_estimators', 50, 200),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'scale_pos_weight': scale_pos_weight,
        'random_state': SEED,
        'verbose': -1,
    }
    model = lgb.LGBMClassifier(**params)
    model.fit(X_train, y_train)
    y_proba = model.predict_proba(X_val)[:, 1]
    pr_auc = average_precision_score(y_val, y_proba)
    return pr_auc

mlflow.set_experiment("hospital-readmission-lightgbm")
mlflow_callback_lgb = MLflowCallback(
    tracking_uri=mlflow.get_tracking_uri(),
    metric_name='pr_auc',
)

study_lgb = optuna.create_study(direction='maximize', study_name='lightgbm-optuna')
study_lgb.optimize(objective_lgb, n_trials=5, callbacks=[mlflow_callback_lgb], show_progress_bar=True)
print(f'Best LightGBM PR-AUC: {study_lgb.best_value:.4f}')

2026/03/10 21:38:19 INFO mlflow.tracking.fluent: Experiment with name 'hospital-readmission-lightgbm' does not exist. Creating a new experiment.
/var/folders/_y/x_pznj0x0m16xm40mxczxrl00000gn/T/ipykernel_44369/1055481184.py:19: ExperimentalWarning: MLflowCallback is experimental (supported from v1.4.0). The interface can change in the future.
  mlflow_callback_lgb = MLflowCallback(
[I 2026-03-10 21:38:19,206] A new study created in memory with name: lightgbm-optuna


  0%|          | 0/5 [00:00<?, ?it/s]

/Users/isabelwu/miniconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
2026/03/10 21:38:19 INFO mlflow.tracking.fluent: Experiment with name 'lightgbm-optuna' does not exist. Creating a new experiment.


[I 2026-03-10 21:38:19,405] Trial 0 finished with value: 0.2029436117538158 and parameters: {'learning_rate': 0.017875291647420907, 'max_depth': 3, 'n_estimators': 74, 'subsample': 0.6958808542773346, 'colsample_bytree': 0.6514595487544743}. Best is trial 0 with value: 0.2029436117538158.
🏃 View run 0 at: http://127.0.0.1:5001/#/experiments/5/runs/7bb9cb8d7cd54edfa606bf5829077cf8
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/5


/Users/isabelwu/miniconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


...

[I 2026-03-10 21:38:20,949] Trial 2 finished with value: 0.18434499685383454 and parameters: {'learning_rate': 0.23636646030671546, 'max_depth': 8, 'n_estimators': 194, 'subsample': 0.9562679604249359, 'colsample_bytree': 0.7817695303246714}. Best is trial 0 with value: 0.2029436117538158.
🏃 View run 2 at: http://127.0.0.1:5001/#/experiments/5/runs/6d932e1870b34de08c8b857a3bed0f07
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/5


/Users/isabelwu/miniconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[I 2026-03-10 21:38:21,258] Trial 3 finished with value: 0.20540470571099417 and parameters: {'learning_rate': 0.16122212689404605, 'max_depth': 5, 'n_estimators': 54, 'subsample': 0.6414257004035134, 'colsample_bytree': 0.791279972568102}. Best is trial 3 with value: 0.20540470571099417.
🏃 View run 3 at: http://127.0.0.1:5001/#/experiments/5/runs/0790b47506804cda833fcce2db0181b4
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/5


/Users/isabelwu/miniconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[I 2026-03-10 21:38:21,748] Trial 4 finished with value: 0.2135906133272142 and parameters: {'learning_rate': 0.030582908250030107, 'max_depth': 8, 'n_estimators': 58, 'subsample': 0.7695000993063091, 'colsample_bytree': 0.8660139016042003}. Best is trial 4 with value: 0.2135906133272142.
🏃 View run 4 at: http://127.0.0.1:5001/#/experiments/5/runs/209d3cdae5d64af294b579897f34b32d
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/5
Best LightGBM PR-AUC: 0.2136


## Log Best Model & Promote if Beats Baseline
Train best config, log full metrics (PR-AUC, ROC-AUC, Recall@K20), confusion matrix, and register model if it beats baseline.

In [6]:
BASELINE_PR_AUC = 0.189  # From scenario-1

best_xgb = study_xgb.best_params
best_lgb = study_lgb.best_params

# Pick champion by PR-AUC
champion_pr_auc = max(study_xgb.best_value, study_lgb.best_value)
if study_xgb.best_value >= study_lgb.best_value:
    champion_name, champion_params = 'XGBoost', best_xgb
    champion_model = xgb.XGBClassifier(scale_pos_weight=scale_pos_weight, random_state=SEED, **best_xgb)
else:
    champion_name, champion_params = 'LightGBM', best_lgb
    champion_model = lgb.LGBMClassifier(scale_pos_weight=scale_pos_weight, random_state=SEED, verbose=-1, **best_lgb)

champion_model.fit(X_train, y_train)
y_proba_val = champion_model.predict_proba(X_val)[:, 1]
y_pred_val = champion_model.predict(X_val)

pr_auc = average_precision_score(y_val, y_proba_val)
roc_auc = roc_auc_score(y_val, y_proba_val)
rec_k20 = recall_at_k(y_val, y_proba_val, k=0.2)

mlflow.set_experiment("hospital-readmission-champion")
with mlflow.start_run(run_name=f"champion_{champion_name}") as run:
    mlflow.log_params(champion_params)
    mlflow.log_param("model_type", champion_name)
    mlflow.log_metric("pr_auc", pr_auc)
    mlflow.log_metric("roc_auc", roc_auc)
    mlflow.log_metric("recall_at_k20", rec_k20)
    mlflow.set_tag("beats_baseline", pr_auc > BASELINE_PR_AUC)
    
    cm = confusion_matrix(y_val, y_pred_val)
    cm_df = pd.DataFrame(cm, index=['No', 'Yes'], columns=['No', 'Yes'])
    cm_df.to_csv("confusion_matrix_champion.csv")
    mlflow.log_artifact("confusion_matrix_champion.csv")
    
    import seaborn as sns
    import matplotlib.pyplot as plt
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Confusion Matrix ({champion_name} Champion)')
    plt.ylabel('True')
    plt.xlabel('Predicted')
    plt.tight_layout()
    plt.savefig('confusion_matrix_champion.png')
    plt.close()
    mlflow.log_artifact('confusion_matrix_champion.png')
    
    input_example = X_val[0:1]
    if hasattr(input_example, 'toarray'):
        input_example = input_example.toarray()
    if champion_name == 'XGBoost':
        mlflow.xgboost.log_model(champion_model, "model", input_example=input_example)
    else:
        mlflow.lightgbm.log_model(champion_model, "model", input_example=input_example)
    
    if pr_auc > BASELINE_PR_AUC:
        mlflow.register_model(f"runs:/{run.info.run_id}/model", "hospital-readmission-risk")
        print(f'Champion promoted: PR-AUC={pr_auc:.3f} > baseline {BASELINE_PR_AUC}')
    else:
        print(f'Champion NOT promoted: PR-AUC={pr_auc:.3f} <= baseline {BASELINE_PR_AUC}')
    print(f'Run ID: {run.info.run_id}')

/Users/isabelwu/miniconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
2026/03/10 21:38:22 INFO mlflow.tracking.fluent: Experiment with name 'hospital-readmission-champion' does not exist. Creating a new experiment.
2026/03/10 21:38:23 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
/Users/isabelwu/miniconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
2026/03/10 21:38:23 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see

Champion promoted: PR-AUC=0.214 > baseline 0.189
Run ID: d76b7696061344dcbe0d5abd7bc331a1
🏃 View run champion_LightGBM at: http://127.0.0.1:5001/#/experiments/6/runs/d76b7696061344dcbe0d5abd7bc331a1
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/6


Created version '1' of model 'hospital-readmission-risk'.
